<a id="setup"></a>
# Entraînement final, réglage des hyperparamètres et inférence

Ce notebook rassemble tout ce qui a été construit dans 01 à 05 pour produire la version finale du modèle : une boucle d'entraînement/validation complète, une petite recherche d'hyperparamètres, une évaluation approfondie qui identifie explicitement les tags les plus difficiles, et un module d'inférence propre et réutilisable en dehors de Jupyter.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from bambara_pos_utils import *

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)

In [ ]:
# Chargement du corpus et split reproductible (même seed que le notebook 05)
corpus_path = Path.cwd().parent / "04_bambara_pos_pipeline" / "bambara_pos_prep_retagged.conll"
corpus_data = load_bambara_corpus(corpus_path)

train_data, val_data, test_data = split_corpus(corpus_data, seed=42)
print(f"Train : {len(train_data)} | Val : {len(val_data)} | Test : {len(test_data)}")

word_to_ix, _ = build_vocab(train_data)
print(f"Taille du vocabulaire (train uniquement) : {len(word_to_ix)}")

<a id="full-training-loop"></a>
# Boucle d'entraînement et de validation complète

La version précédente (notebook 05) entraînait sans suivre la validation en continu ni sauvegarder de checkpoint. Ici, on ajoute trois choses qui manquaient pour un vrai pipeline de fine-tuning :

Suivi de la loss et de l'accuracy sur validation à chaque époque, pas seulement à la fin.
Sauvegarde du meilleur modèle (bambara_pos_best.pth), basée sur la loss de validation — pas sur la dernière époque, qui n'est pas forcément la meilleure.
Arrêt anticipé (early stopping) simple : si la validation ne s'améliore plus pendant plusieurs époques, on arrête, ce qui évite de sur-entraîner inutilement et fait gagner du temps de calcul.

In [ ]:
def run_training(train_data, val_data, word_to_ix, hyperparams, device,
                  patience=3, max_epochs=30, checkpoint_path=None, verbose=True):
    """
    Boucle d'entraînement/validation complète avec class weights, early stopping
    et sauvegarde du meilleur modèle. Retourne le modèle, l'historique des métriques
    et la meilleure loss de validation atteinte.
    """
    train_loader = DataLoader(
        BambaraPOSDataset(train_data, word_to_ix, label2id),
        batch_size=hyperparams["batch_size"], shuffle=True, collate_fn=collate_fn_padd
    )
    val_loader = DataLoader(
        BambaraPOSDataset(val_data, word_to_ix, label2id),
        batch_size=hyperparams["batch_size"], shuffle=False, collate_fn=collate_fn_padd
    )

    model = LSTMTaggerWithBatch(
        hyperparams["embedding_dim"], hyperparams["hidden_dim"],
        len(word_to_ix), len(label2id)
    ).to(device)

    class_weights = get_class_weights(train_data, label2id, device)
    loss_fn = nn.NLLLoss(weight=class_weights, ignore_index=0)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=hyperparams["learning_rate"], weight_decay=0.01
    )

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, len(label2id), device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if verbose:
            print(f"Époque {epoch+1:2d} | Train Loss : {train_loss:.4f} | "
                  f"Val Loss : {val_loss:.4f} | Val Acc : {val_acc:.2f}%")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            if checkpoint_path is not None:
                save_checkpoint(model, word_to_ix, hyperparams, checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                if verbose:
                    print(f"Arrêt anticipé à l'époque {epoch+1} (pas d'amélioration depuis {patience} époques).")
                break

    return model, history, best_val_loss


# get_class_weights réutilisée du notebook 05 : on la redéfinit ici pour que ce
# notebook reste exécutable seul, sans dépendre de 05 (qui a son propre kernel).
def get_class_weights(train_data, label2id, device):
    train_labels_flat = [label2id[t] for _, tags in train_data for t in tags]
    unique_labels = np.unique(train_labels_flat)
    weights = compute_class_weight(class_weight="balanced", classes=unique_labels, y=train_labels_flat)
    class_weights = torch.ones(len(label2id), dtype=torch.float)
    for label_idx, weight in zip(unique_labels, weights):
        class_weights[label_idx] = weight
    class_weights[0] = 0.0
    return class_weights.to(device)

<a id="hyperparam-tuning"></a>
# Recherche d'hyperparamètres

On teste un petit nombre de combinaisons raisonnables plutôt qu'une grille exhaustive — avec 37 392 phrases, une recherche exhaustive prendrait beaucoup trop de temps pour ce que ça apporterait. On fait varier trois hyperparamètres qui ont le plus d'impact connu sur ce type de modèle : le taux d'apprentissage, la taille de batch, et la taille de la couche cachée. Chaque configuration est entraînée sur peu d'époques (avec early stopping) juste pour comparer, pas pour produire le modèle final.

In [ ]:
import itertools
import time

search_space = {
    "learning_rate": [1e-3, 5e-4],
    "batch_size": [16, 32, 64],
    "hidden_dim": [64, 128],
}
FIXED_EMBEDDING_DIM = 100

combinations = list(itertools.product(
    search_space["learning_rate"], search_space["batch_size"], search_space["hidden_dim"]
))
print(f"{len(combinations)} configurations à tester.")

results = []
for lr, bs, hd in combinations:
    hp = {"learning_rate": lr, "batch_size": bs, "hidden_dim": hd, "embedding_dim": FIXED_EMBEDDING_DIM}
    print(f"\n--- lr={lr} | batch_size={bs} | hidden_dim={hd} ---")

    start = time.time()
    _, history, best_val_loss = run_training(
        train_data, val_data, word_to_ix, hp, device,
        patience=2, max_epochs=6, checkpoint_path=None, verbose=False
    )
    elapsed = time.time() - start

    results.append({
        **hp,
        "best_val_loss": best_val_loss,
        "final_val_acc": history["val_acc"][-1],
        "epochs_run": len(history["train_loss"]),
        "time_sec": round(elapsed, 1),
    })
    print(f"Val Loss : {best_val_loss:.4f} | Val Acc : {history['val_acc'][-1]:.2f}% | "
          f"Temps : {elapsed:.1f}s")

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).sort_values("best_val_loss")
results_df

On choisit la configuration avec la meilleure best_val_loss (pas la meilleure accuracy brute, qui peut être trompeuse sur un jeu de validation encore déséquilibré). Le tableau ci-dessus permet aussi de repérer les tendances : par exemple si un hidden_dim plus grand n'améliore presque rien, ce n'est pas la peine d'alourdir le modèle pour rien.

In [ ]:
best_config = results_df.iloc[0].to_dict()
best_hyperparams = {
    "learning_rate": best_config["learning_rate"],
    "batch_size": int(best_config["batch_size"]),
    "hidden_dim": int(best_config["hidden_dim"]),
    "embedding_dim": FIXED_EMBEDDING_DIM,
}
print("Meilleure configuration retenue :", best_hyperparams)

In [ ]:
# Entraînement final avec la meilleure configuration, cette fois sans limiter les époques
checkpoint_path = MODELS_DIR / "bambara_pos_best.pth"

final_model, final_history, final_best_val_loss = run_training(
    train_data, val_data, word_to_ix, best_hyperparams, device,
    patience=4, max_epochs=40, checkpoint_path=checkpoint_path, verbose=True
)

print(f"\nMeilleur modèle sauvegardé dans : {checkpoint_path}")

In [ ]:
# Courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(final_history["train_loss"], label="Train")
axes[0].plot(final_history["val_loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(final_history["val_acc"], color="green")
axes[1].set_title("Accuracy de validation (%)")
axes[1].set_xlabel("Époque")

plt.tight_layout()
plt.show()

<a id="final-evaluation"></a>
# Évaluation finale et tags les plus difficiles

On recharge le meilleur checkpoint (pas forcément le modèle en mémoire, pour être sûr d'évaluer exactement ce qui a été sauvegardé) et on l'évalue sur le test set, jamais vu pendant l'entraînement ni la recherche d'hyperparamètres.

In [ ]:
model, loaded_word_to_ix, loaded_hp = load_checkpoint(checkpoint_path, device)
print("Hyperparamètres du modèle chargé :", loaded_hp)

test_loader = DataLoader(
    BambaraPOSDataset(test_data, loaded_word_to_ix, label2id),
    batch_size=32, shuffle=False, collate_fn=collate_fn_padd
)


def evaluate_full(model, dataloader, device, id2label):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = torch.argmax(model(inputs), dim=-1)
            for pred_seq, label_seq in zip(preds.cpu().numpy(), targets.cpu().numpy()):
                for p, l in zip(pred_seq, label_seq):
                    if l != 0:
                        all_preds.append(p)
                        all_labels.append(l)

    accuracy = accuracy_score(all_labels, all_preds)
    present = sorted(set(all_labels) | set(all_preds))
    report_dict = classification_report(
        all_labels, all_preds, labels=present,
        target_names=[id2label[i] for i in present], zero_division=0, output_dict=True
    )
    report_text = classification_report(
        all_labels, all_preds, labels=present,
        target_names=[id2label[i] for i in present], zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds, labels=present)
    return accuracy, report_dict, report_text, cm, present


test_accuracy, report_dict, report_text, test_cm, present_labels = evaluate_full(
    model, test_loader, device, id2label
)

print(f"Accuracy finale sur le test set : {test_accuracy*100:.2f}%\n")
print(report_text)

In [ ]:
# Classement explicite des tags du plus difficile au plus facile pour le modèle
worst_to_best = per_tag_f1(report_dict)

print("Tags classés du plus difficile au plus facile (F1-score) :\n")
for tag, f1 in worst_to_best.items():
    support = report_dict[tag]["support"]
    barre = "#" * int(f1 * 30)
    print(f"  {tag:<8} F1={f1:.3f}  (support={int(support):>6})  {barre}")

### Interprétation

Les tags les plus difficiles ne le sont généralement pas par hasard — chacun a une explication probable qu'il faut vérifier avant de conclure que le modèle est "mauvais" :

Un F1 bas avec un support faible (peu d'exemples dans le test, ex. ADV, PART) reflète surtout un manque de données pour cette classe, pas forcément une faiblesse du modèle — compute_class_weight aide mais ne remplace pas plus d'exemples réels.
Un F1 bas avec un support élevé est plus préoccupant : ça veut dire que le modèle voit beaucoup d'exemples de ce tag et se trompe quand même, souvent parce qu'il est structurellement ambigu avec un autre tag (voir la matrice de confusion du notebook 05 — NOM/VERBE est un candidat probable ici, vu que la règle contextuelle du notebook 04 a pu introduire du bruit sur ces deux tags précisément).
Comme discuté en 05, une partie de la performance globale reste optimiste à cause des labels générés par lexique/règle plutôt que par annotation humaine — les scores par tag ci-dessus héritent de cette même limite, en particulier pour AUX, PRON, POSTP qui sont presque entièrement couverts par le lexique.

In [ ]:
# plot_confusion_matrix(test_cm, present_labels, id2label)  # fonction définie en 05

<a id="inference-module"></a>
# Module d'inférence

Le livrable final n'est pas juste un notebook : c'est un module Python autonome, importable en dehors de Jupyter, qui charge le modèle entraîné et tague une phrase brute. Le code est dans inference.py

In [ ]:
# Démonstration d'utilisation du module d'inférence
from inference import BambaraPOSTagger

tagger = BambaraPOSTagger(checkpoint_path=MODELS_DIR / "bambara_pos_best.pth", device=device)

phrases_test = [
    "Amadou bɛ kalan kɛ sisan .",
    "An ka taa so kɔnɔ .",
    "I ka kɛnɛ wa ?",
]

for phrase in phrases_test:
    resultat = tagger.tag(phrase)
    print(phrase)
    for mot, tag in resultat:
        print(f"   {mot:<12} {tag}")
    print()

### Limites du module d'inférence

Le module hérite de toutes les limites documentées dans ce projet : le lexique couvre uniquement les mots fonctionnels connus, la règle "après un AUX → VERBE" peut se tromper sur des phrases sans verbe explicite (constructions copulatives), et l'accuracy réelle du BiLSTM sur des mots totalement nouveaux est probablement plus proche des chiffres "hors lexique" du notebook 05 que de l'accuracy globale affichée plus haut. Ce module est fonctionnel et utilisable pour des démonstrations et pour continuer le développement